In [1]:
import sqlite3
import pandas as pd
from pathlib import Path
from lifelines import KaplanMeierFitter

DB_PATH = Path('../db/tcga_glioma.db')
conn = sqlite3.connect(DB_PATH)

query = """
SELECT p.patient_id, p.study, p.histology, p.grade, p.age_at_diagnosis,
       p.gender, p.kps, p.mutation_count,
       b.idh_status, b.mgmt_status, b.codel_1p19q,
       s.os_months, s.os_event
FROM patients p
JOIN biomarkers b ON p.patient_id = b.patient_id
JOIN survival s ON p.patient_id = s.patient_id;
"""
df = pd.read_sql_query(query, conn)

# Te same decyzje o jakości danych co w Etapach 4-5
df = df[df['os_months'] >= 0].reset_index(drop=True)
df_km = df.dropna(subset=['os_event']).copy()

print(df_km.shape)

(1043, 13)


In [3]:
# Eksport 1: dane pacjentów 
df_km.to_csv('../data/processed/tableau_patients.csv', index=False)
print("Zapisano tableau_patients.csv:", df_km.shape)

Zapisano tableau_patients.csv: (1043, 13)


In [4]:
def km_to_long(kmf, grupa, typ_podzialu):
    sf = kmf.survival_function_.reset_index()
    sf.columns = ['time', 'survival_prob']
    sf['grupa'] = grupa
    sf['typ_podzialu'] = typ_podzialu
    return sf

wyniki = []

# Cała kohorta
kmf_all = KaplanMeierFitter()
kmf_all.fit(df_km['os_months'], df_km['os_event'])
wyniki.append(km_to_long(kmf_all, 'Cała kohorta', 'Cała kohorta'))

# Podział wg IDH
df_idh = df_km.dropna(subset=['idh_status'])
for status in ['Mutant', 'WT']:
    subset = df_idh[df_idh['idh_status'] == status]
    kmf_tmp = KaplanMeierFitter()
    kmf_tmp.fit(subset['os_months'], subset['os_event'])
    wyniki.append(km_to_long(kmf_tmp, f'IDH {status}', 'IDH'))

# Podział wg MGMT
df_mgmt = df_km.dropna(subset=['mgmt_status'])
for status in ['Methylated', 'Unmethylated']:
    subset = df_mgmt[df_mgmt['mgmt_status'] == status]
    kmf_tmp = KaplanMeierFitter()
    kmf_tmp.fit(subset['os_months'], subset['os_event'])
    wyniki.append(km_to_long(kmf_tmp, f'MGMT {status}', 'MGMT'))

# 4 grupy IDH x MGMT
df_combo = df_km.dropna(subset=['idh_status', 'mgmt_status']).copy()
df_combo['grupa_combo'] = df_combo['idh_status'] + ' / ' + df_combo['mgmt_status']
for grupa in df_combo['grupa_combo'].unique():
    subset = df_combo[df_combo['grupa_combo'] == grupa]
    kmf_tmp = KaplanMeierFitter()
    kmf_tmp.fit(subset['os_months'], subset['os_event'])
    wyniki.append(km_to_long(kmf_tmp, grupa, 'IDH x MGMT'))

km_export = pd.concat(wyniki, ignore_index=True)
km_export.to_csv('../data/processed/tableau_survival_curves.csv', index=False)
print("Zapisano tableau_survival_curves.csv:", km_export.shape)
km_export.head()

Zapisano tableau_survival_curves.csv: (2821, 4)


,time,survival_prob,grupa,typ_podzialu
0,0.000000,1.000000,Cała kohorta,Cała kohorta
1,0.032855,1.000000,Cała kohorta,Cała kohorta
2,0.065710,1.000000,Cała kohorta,Cała kohorta
3,0.098565,0.999036,Cała kohorta,Cała kohorta
4,0.131420,0.999036,Cała kohorta,Cała kohorta
